# 1-3 LangChain 프레임워크 깊이 있는 학습

원본 슬라이드: [1-3-langchain-framework-deep-dive.pdf](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf)  
이 노트북은 슬라이드의 개념 순서를 따라가는 학습용 해설과 작은 실험입니다. 원본 PDF는 그대로 보존합니다.

## 학습 목표
- LCEL 체인과 Runnable의 실행 방식을 읽고 설명한다.
- 도구 제안·파싱·실행·관찰의 책임을 구분한다.
- 구조화 출력과 메모리 전략을 작업 목적에 맞게 비교한다.

**실습 방식:** 예제를 먼저 실행해 관찰하고, CodeMate에 개선을 요청한 뒤 생성 결과를 직접 검토합니다. 실제 CodeMate 메뉴·버튼 이름은 강의장 환경에서 확인하세요.


## 슬라이드별 찾아보기

각 쪽의 소제목과 첫 핵심 문구를 찾아가기 위한 표입니다. 본문의 설명·실습과 함께 읽고, 원본의 그림·도표·화면 예시는 쪽 번호를 눌러 확인하세요. PDF에서 수식 또는 그림 안 글자가 추출되지 않는 쪽은 원본 화면을 기준으로 읽습니다.

| PDF 쪽 | 소제목 | 확인할 내용 |
|---|---|---|
| [1쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=1) | 1-3 | LangChain 프레임워크 깊이 있는 학습 |
| [2쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=2) | LCEL과 Runnable | LCEL의 설계 철학 |
| [3쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=3) | LCEL과 Runnable | 원본 슬라이드의 그림·배치 확인 |
| [4쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=4) | LCEL의 설계 철학 (1) | LCEL은 LLM 애플리케이션을 작은 실행 단위의 조합으로 표현하기 위한 LangChain의 체인 구성 방식이다. |
| [5쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=5) | LCEL의 설계 철학 (2) | LCEL의 핵심 장점은 실행 흐름을 선언적으로 읽을 수 있다는 점이다. |
| [6쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=6) | Runnable 인터페이스 (1) | Runnable은 LangChain에서 실행 가능한 구성 요소를 공통 방식으로 다루기 위한 인터페이스이다. |
| [7쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=7) | Runnable 인터페이스 (2) | Runnable 조합은 복잡한 입력 변환을 단계별로 나누는 데 유용하다. |
| [8쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=8) | Tool Calling | 원본 슬라이드의 그림·배치 확인 |
| [9쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=9) | Tool Schema와 도구 선택 (1) | Tool Calling은 LLM이 자연어 답변 대신 사용할 도구와 인자를 구조화된 형태로 선택하게 하는 방식이다. |
| [10쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=10) | 응답 파싱과 실행 흐름 (1) | Tool Calling의 실행은 모델이 도구를 직접 실행하는 것이 아니라, 모델의 도구 호출 의도를 외부 런타임이 실행하는 |
| [11쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=11) | 응답 파싱과 실행 흐름 (2) | Agent의 기본 실행 흐름은 Tool Calling을 반복적으로 활용한다. |
| [12쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=12) | 구조화된 출력과 메모리 | 원본 슬라이드의 그림·배치 확인 |
| [13쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=13) | Output Parser와 Pydantic | Output Parser는 LLM의 텍스트 출력을 애플리케이션이 사용할 수 있는 구조화된 데이터로 바꾸는 역할을 한다. |
| [14쪽](../../../raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf#page=14) | Buffer, Summary, VectorStoreRetriever Memory | Memory는 대화나 작업 과정에서 이전 정보를 다음 요청에 반영하기 위한 설계이다. |


## 1. LCEL과 Runnable (슬라이드 3–7)

LCEL은 프롬프트, 모델, 파서 같은 작은 실행 단위를 `|`로 연결하는 방식입니다. 앞 단계 출력이 다음 단계 입력이 됩니다. Runnable은 `invoke`, `batch`, `stream`, `ainvoke` 같은 공통 실행 방식을 제공합니다. `RunnablePassthrough`는 입력 전달, `RunnableLambda`는 함수 감싸기, `RunnableParallel`은 여러 결과 묶기에 쓰입니다.

아래 코드는 외부 모델 없이 **파이프라인의 데이터 흐름만** 보여줍니다. 실제 LCEL API 예시는 PDF 슬라이드 4–7을 보세요.

In [ ]:
def make_prompt(topic):
    return f"{topic}를 입문자에게 한 문장으로 설명해 주세요."
def mock_model(prompt):
    return f"[모의 응답] 요청 내용: {prompt}"
def parse_text(response):
    return response.strip()

steps = (make_prompt, mock_model, parse_text)
value = "Runnable"
for step in steps:
    value = step(value)
    print(step.__name__, "→", value)

### 슬라이드 코드와 실습 코드 연결 (슬라이드 4, 6–7)

원본의 `prompt | model | parser`에서 위의 `make_prompt → mock_model → parse_text`가 각각 같은 위치에 있습니다. 실제 LangChain에서는 다음과 같이 연결합니다. 모델 연결·자격증명이 필요한 코드는 강의장 환경에서 확인한 뒤 실행하세요.

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("{topic}을 초보자에게 설명해줘.")
# model = 강의장에서 설정한 ChatModel
# chain = prompt | model | StrOutputParser()
# chain.invoke({"topic": "LCEL"})
# chain.batch([{"topic": "Runnable"}, {"topic": "Tool Calling"}])
# for chunk in chain.stream({"topic": "Agent"}): ...
```

슬라이드 7의 `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`은 검색 결과·질문·메타데이터를 **하나의 프롬프트 입력 묶음**으로 만드는 예입니다. 어느 값이 어느 키에 들어가는지 출력 형태를 먼저 확인하세요.

**실제 코드 읽기:** PDF 4쪽의 `prompt | model | StrOutputParser()`에서 각 단계의 입력·출력을 표로 적어 보세요. 실제 LLM 호출에는 강의장 모델 연결과 자격증명이 필요합니다.

## 2. Tool Calling (슬라이드 8–11)

Tool Schema는 도구 이름·설명·인자 타입을 모델에 알려줍니다. 모델은 도구 호출 **의도**를 만들고, 애플리케이션이 이를 검사·실행합니다. 도구 결과가 다시 모델에 들어가야 최종 답변에 반영됩니다. 반복 Agent에는 도구 선택뿐 아니라 중단 조건도 필요합니다.

In [ ]:
def multiply(a: int, b: int) -> int:
    return a*b

tool_call = {"name": "multiply", "arguments": {"a": 6, "b": 7}}
registry = {"multiply": multiply}
if tool_call["name"] not in registry:
    raise ValueError("알 수 없는 도구")
args = tool_call["arguments"]
if not all(isinstance(args.get(k), int) for k in ("a", "b")):
    raise TypeError("a와 b는 정수여야 합니다")
print("관찰 결과:", registry[tool_call["name"]](**args))

**검토:** `multiply`가 실제 외부 시스템을 조작하는 함수라면 실행 전에 무엇을 더 확인해야 할까요? 도구 설명이 모호하면 선택 품질에 어떤 문제가 생길까요?

## 3. 구조화 출력 (슬라이드 12–13)

일반 텍스트를 다음 프로그램이 안정적으로 사용하려면 정해진 구조와 검증이 필요합니다. Output Parser는 출력 형식을 읽고, Pydantic은 필드·타입·누락을 검사하는 데 쓸 수 있습니다. 형식에 맞는 JSON이라도 내용의 사실성은 별도로 확인해야 합니다.

In [ ]:
import json
raw = '{"tool": "multiply", "arguments": {"a": 6, "b": 7}}'
parsed = json.loads(raw)
assert set(parsed) == {"tool", "arguments"}
assert isinstance(parsed["arguments"]["a"], int)
print(parsed)

## 4. 메모리 전략 (슬라이드 14)

- **Buffer:** 이전 대화를 그대로 유지. 짧은 대화에서 단순하지만 컨텍스트가 커집니다.
- **Summary:** 이전 대화를 요약해 토큰을 아낍니다. 요약에서 빠진 정보는 복구하기 어렵습니다.
- **VectorStoreRetriever:** 과거 기록 중 관련 항목만 검색합니다. 검색 실패 가능성을 평가해야 합니다.

어떤 방식이든 “저장된 정보가 다음 요청에 어떻게 들어오는가”를 확인하세요.

### CodeMate 페어 실습

> 위 `multiply` 예제를 도구 스키마, 인자 검증, 실패 메시지, 실행 기록을 가진 작은 런타임으로 바꿔 주세요. 먼저 입력·출력 계약과 실패 사례 3가지를 써 주세요. 모델 호출은 하지 마세요.

**리뷰 질문:** 알 수 없는 도구, 빠진 인자, 잘못된 타입을 모두 잡나요? 실행 기록만 보고 어떤 도구가 실행됐는지 알 수 있나요?

## 정리

1. `invoke`와 `batch`는 어떤 입력 상황에서 각각 쓰나요?
2. 모델의 도구 제안과 실제 도구 실행은 누가 담당하나요?
3. 긴 대화에서 Summary Memory를 쓰면 어떤 정보 손실이 생길 수 있나요?